In [12]:
import pandas as pd
import numpy as np
import networkx as nx

# =============================================================================
# CONFIGURATION
# =============================================================================
FILE_PATH = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results/identified_isotopes.csv"
OUTPUT_PATH = "/home/ajarrah/PhD_Thesis/chapter_4/code_final/2_mz_synced_isotope_80_matching_results/parent_children_hierarchy_network.csv"

def build_network_hierarchy(path):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"Error: Could not find file at {path}")
        return None

    # 1. Create a Graph where each m/z is a node and each row is an edge
    # This automatically handles complex relationships (A-B, B-C -> A-B-C)
    G = nx.Graph()
    for _, row in df.iterrows():
        G.add_edge(row['mz_1'], row['mz_2'])

    # 2. Extract connected components (families)
    families = list(nx.connected_components(G))
    
    rows = []
    max_child_count = 0
    
    # 3. Process each family
    for family in families:
        # Sort the m/z values in the family
        sorted_mzs = sorted(list(family))
        
        # We define the smallest m/z in the group as the "Parent"
        parent = sorted_mzs[0]
        # Everything else is a child
        children = sorted_mzs[1:]
        
        max_child_count = max(max_child_count, len(children))
        
        row_dict = {'Parent_MZ': parent}
        for i, child_mz in enumerate(children):
            row_dict[f'Child_{i+1}'] = child_mz
            
        rows.append(row_dict)

    # 4. Create DataFrame and sort by Parent MZ
    final_df = pd.DataFrame(rows)
    final_df = final_df.sort_values(by='Parent_MZ').reset_index(drop=True)
    
    # Organize columns
    column_order = ['Parent_MZ'] + [f'Child_{i+1}' for i in range(max_child_count)]
    
    # Ensure all columns exist (fill with NaN if a family is small)
    for col in column_order:
        if col not in final_df.columns:
            final_df[col] = np.nan
            
    return final_df[column_order]

# =============================================================================
# EXECUTION
# =============================================================================
result_table = build_network_hierarchy(FILE_PATH)

if result_table is not None:
    print(f"\nSuccessfully grouped {len(result_table)} related families.")
    print("Preview of the sorted hierarchical table (Strict Row-Based Relationships):")
    print(result_table.head())
    
    # Save to the specified path
    result_table.to_csv(OUTPUT_PATH, index=False)
    print(f"\nFinal table saved to: {OUTPUT_PATH}")


Successfully grouped 83 related families.
Preview of the sorted hierarchical table (Strict Row-Based Relationships):
   Parent_MZ   Child_1   Child_2   Child_3   Child_4   Child_5   Child_6  \
0   339.2312  340.2334  351.2299  352.2358  353.2464  354.2528  355.2620   
1   363.1714  364.1751  365.1871       NaN       NaN       NaN       NaN   
2   370.2467  371.2570  372.2607       NaN       NaN       NaN       NaN   
3   379.1659  380.1739  381.1827  382.1865  383.1968  384.2020  385.2119   
4   386.2417  387.2512  388.2554  389.2662       NaN       NaN       NaN   

    Child_7   Child_8   Child_9  ...  Child_91  Child_92  Child_93  Child_94  \
0  356.2652  357.2677  358.2712  ...       NaN       NaN       NaN       NaN   
1       NaN       NaN       NaN  ...       NaN       NaN       NaN       NaN   
2       NaN       NaN       NaN  ...       NaN       NaN       NaN       NaN   
3  386.2166       NaN       NaN  ...       NaN       NaN       NaN       NaN   
4       NaN       NaN    